# Import Library & Import Data

In [1]:
# Data Processing
import pandas as pd
import numpy as np

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Datetime Standardize
import datetime

In [2]:
# Import Library & Import Data
import sys
import matplotlib

print(f"Python version: {sys.version.split()[0]}")
print(f"pandas version: {pd.__version__}")
print(f"matplotlib version: {matplotlib.__version__}")
print(f"numpy version: {np.__version__}")

Python version: 3.13.16
pandas version: 2.2.3
matplotlib version: 3.10.0
numpy version: 2.1.3


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# Định nghĩa đường dẫn
base_path = '/content/drive/MyDrive/K04 - Scripts các buổi học/Dataset/Buổi 8: Funnel Chart/'

# Load dữ liệu
order_items = pd.read_csv(base_path + 'order_items.csv')
refunds     = pd.read_csv(base_path + 'order_item_refunds.csv')
products    = pd.read_csv(base_path + 'products.csv')
pageviews   = pd.read_csv(base_path + 'website_pageviews.csv')
sessions    = pd.read_csv(base_path + 'website_sessions.csv')
orders      = pd.read_csv(base_path + 'orders.csv')

# Kiểm tra nhanh dữ liệu chi tiết đơn hàng
print("Dữ liệu chi tiết đơn hàng (order_items):")
display(order_items.head())

Dữ liệu chi tiết đơn hàng (order_items):


,order_item_id,created_at,order_id,product_id,is_primary_item,price_usd,cogs_usd
0,1,2012-03-19 10:42:46,1,1,1,49.99,19.49
1,2,2012-03-19 19:27:37,2,1,1,49.99,19.49
2,3,2012-03-20 06:44:45,3,1,1,49.99,19.49
3,4,2012-03-20 09:41:45,4,1,1,49.99,19.49
4,5,2012-03-20 11:28:15,5,1,1,49.99,19.49


# Data Checking

# Data Preprocessing / Wrangling

# Exploratory - EDA (Đưa vào phương pháp phân tích or thống kê)

In [6]:
orders_col_df = orders[['order_id', 'created_at', 'user_id']].drop_duplicates(ignore_index = True)
order_items_col_df = order_items[['order_id', 'product_id']].drop_duplicates(ignore_index = True)
products_col_df = products[['product_id', 'product_name']].drop_duplicates(ignore_index = True)
df_merge = orders_col_df.merge(order_items_col_df, on = 'order_id', how = 'inner')
df_prep = df_merge.merge(products_col_df, on = 'product_id' , how = 'inner')
df_pre_sankey = df_prep[['created_at', 'user_id', 'product_name']].drop_duplicates(ignore_index = True)
df_pre_sankey

,created_at,user_id,product_name
0,2012-03-19 10:42:46,20,The Original Mr. Fuzzy
1,2012-03-19 19:27:37,104,The Original Mr. Fuzzy
2,2012-03-20 06:44:45,147,The Original Mr. Fuzzy
3,2012-03-20 09:41:45,160,The Original Mr. Fuzzy
4,2012-03-20 11:28:15,177,The Original Mr. Fuzzy
...,...,...,...
40020,2015-03-19 04:10:43,394257,The Hudson River Mini bear
40021,2015-03-19 05:27:28,394268,The Forever Love Bear
40022,2015-03-19 05:27:28,394268,The Hudson River Mini bear
40023,2015-03-19 05:35:57,394273,The Hudson River Mini bear


In [8]:
# Unit test
df_test = df_pre_sankey[df_pre_sankey['user_id'] == 173069]
df_test

,created_at,user_id,product_name
13738,2014-03-24 10:21:46,173069,The Forever Love Bear
13739,2014-03-24 10:21:46,173069,The Hudson River Mini bear
14993,2014-04-18 09:45:23,173069,The Original Mr. Fuzzy
14994,2014-04-18 09:45:23,173069,The Birthday Sugar Panda


In [10]:
# Sắp xếp lại dữ liệu
df_sankey_prep_order = df_test.sort_values(by = 'created_at', ascending = True).reset_index(drop = True)
# Tím rank để biết đây là sản phẩm thứ mấy? mà khách hàng mua
df_sankey_prep_order['rank'] = df_sankey_prep_order.groupby(by = 'user_id')['created_at'].rank(method = 'first').astype(int)
# Ghép chuỗi
df_sankey_prep_order['product_name_rank'] = df_sankey_prep_order['product_name'].astype(str) + ' - ' + df_sankey_prep_order['rank'].astype(str)
# Tìm sản phẩm mà khách hàng mua tiếp theo sau sản phẩm của DÒNG hiện tại
df_sankey_prep_order['next_product_name_rank'] = df_sankey_prep_order.groupby(by = 'user_id')['product_name_rank'].shift(-1)

# Điền "End" vào bản ghi bị rỗng
df_sankey_prep_order['next_product_name_rank'] = df_sankey_prep_order['next_product_name_rank'].fillna('End')

# In
df_sankey_prep_order

,created_at,user_id,product_name,rank,product_name_rank,next_product_name_rank
0,2014-03-24 10:21:46,173069,The Forever Love Bear,1,The Forever Love Bear - 1,The Hudson River Mini bear - 2
1,2014-03-24 10:21:46,173069,The Hudson River Mini bear,2,The Hudson River Mini bear - 2,The Original Mr. Fuzzy - 3
2,2014-04-18 09:45:23,173069,The Original Mr. Fuzzy,3,The Original Mr. Fuzzy - 3,The Birthday Sugar Panda - 4
3,2014-04-18 09:45:23,173069,The Birthday Sugar Panda,4,The Birthday Sugar Panda - 4,End


In [12]:
# Sắp xếp lại dữ liệu
df_sankey_prep_order = df_pre_sankey.sort_values(by = 'created_at', ascending = True).reset_index(drop = True)
# Tím rank để biết đây là sản phẩm thứ mấy? mà khách hàng mua
df_sankey_prep_order['rank'] = df_sankey_prep_order.groupby(by = 'user_id')['created_at'].rank(method = 'first').astype(int)
# Ghép chuỗi
df_sankey_prep_order['product_name_rank'] = df_sankey_prep_order['product_name'].astype(str) + ' - ' + df_sankey_prep_order['rank'].astype(str)
# Tìm sản phẩm mà khách hàng mua tiếp theo sau sản phẩm của DÒNG hiện tại
df_sankey_prep_order['next_product_name_rank'] = df_sankey_prep_order.groupby(by = 'user_id')['product_name_rank'].shift(-1)
# Điền "End" vào bản ghi bị rỗng
df_sankey_prep_order['next_product_name_rank'] = df_sankey_prep_order['next_product_name_rank'].fillna('End')
# Tổng hợp dữ liệu
df_sankey = df_sankey_prep_order.groupby(by = ['product_name_rank', 'next_product_name_rank'], as_index = False).agg(count_user = ('user_id', 'nunique'))

#In
df_sankey

,product_name_rank,next_product_name_rank,count_user
0,The Birthday Sugar Panda - 1,End,1957
1,The Birthday Sugar Panda - 1,The Birthday Sugar Panda - 2,10
2,The Birthday Sugar Panda - 1,The Forever Love Bear - 2,116
3,The Birthday Sugar Panda - 1,The Hudson River Mini bear - 2,481
4,The Birthday Sugar Panda - 1,The Original Mr. Fuzzy - 2,654
...,...,...,...
63,The Original Mr. Fuzzy - 3,The Hudson River Mini bear - 4,20
64,The Original Mr. Fuzzy - 3,The Original Mr. Fuzzy - 4,1
65,The Original Mr. Fuzzy - 4,End,19
66,The Original Mr. Fuzzy - 4,The Birthday Sugar Panda - 5,2


In [13]:
import plotly.graph_objects as go
import pandas as pd
import re

# (Giả sử df_sankey của bạn đã có sẵn từ bước trước)
# df_sankey = ...

# 1. Lấy danh sách tất cả các node duy nhất
all_nodes = sorted(list(set(df_sankey['product_name_rank']).union(set(df_sankey['next_product_name_rank']))))
node_indices = {node: i for i, node in enumerate(all_nodes)}

# 2. Định nghĩa bảng màu cho từng nhóm rank (bạn có thể thay đổi mã màu tùy ý)
# Ví dụ: Nhóm 1 màu xanh dương, Nhóm 2 màu cam, Nhóm 3 màu xanh lá, End màu xám,...
color_palette = {
    '1': 'rgba(31, 119, 180, 0.8)',   # Nhóm -1 (Xanh dương)
    '2': 'rgba(255, 127, 14, 0.8)',   # Nhóm -2 (Cam)
    '3': 'rgba(44, 160, 44, 0.8)',    # Nhóm -3 (Xanh lá)
    '4': 'rgba(214, 39, 40, 0.8)',    # Nhóm -4 (Đỏ)
    'End': 'rgba(127, 127, 127, 0.8)' # Nút End (Xám)
}
default_color = 'rgba(148, 103, 189, 0.8)' # Màu dự phòng nếu có rank lớn hơn

# 3. Hàm tự động gán màu cho từng node dựa vào số đuôi (rank)
def get_node_color(node_name):
    if node_name == 'End':
        return color_palette['End']

    # Tìm số rank ở cuối chuỗi (ví dụ: " - 1", " - 2")
    match = re.search(r'- (\d+)$', node_name)
    if match:
        rank_num = match.group(1)
        return color_palette.get(rank_num, default_color)
    return default_color

node_colors = [get_node_color(node) for node in all_nodes]

# 4. Xây dựng biểu đồ Sankey
fig = go.Figure(data=[go.Sankey(
    node = dict(
      pad = 15,
      thickness = 20,
      line = dict(color = "black", width = 0.5),
      label = all_nodes,
      color = node_colors  # Áp dụng danh sách màu theo nhóm vào đây
    ),
    link = dict(
      source = df_sankey['product_name_rank'].map(node_indices),
      target = df_sankey['next_product_name_rank'].map(node_indices),
      value = df_sankey['count_user'],
      color = "rgba(200, 200, 200, 0.4)" # Màu luồng liên kết nhạt để làm nổi bật nodes
    ))])

# 5. Cấu hình giao diện
fig.update_layout(
    title_text="<b>Biểu đồ Sankey - Phân nhóm theo thứ tự mua hàng (Rank)</b>",
    font_size=12,
    width=900,
    height=600
)

fig.show()

# Feature Engineering (Optional)

# Conclusion & Recommendation